# Denoise Benchmark Analysis

Reads every `denoise_*.csv` in the current directory and produces two analyses:

1. **Per-version scalability** — one plot per algorithm version, showing how total GPU time scales with image size. Variants (e.g. `denoise_1_4`) appear as separate lines on the version's chart.
2. **Cross-version comparison** — for a chosen image, bar chart comparing all versions.

Naming convention assumed:
- `denoise_<N>.csv` = a distinct algorithm version (naive, two-pass, sorted, final, ...).
- `denoise_<N>_<M>.csv` = a variant of version N (e.g. fast-pass on global memory).
- Image filenames contain a numeric size token (`noise_1280.ppm`, `noise_1920.ppm`, ...) used as the scalability x-axis.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook lives in the same folder as the CSVs.
DATA_DIR = Path('.')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Load and tidy the data

Each CSV is one (executable, image, repetition) measurement. We:

1. Load every `denoise_*.csv`.
2. Drop any row where `status != 'ok'`.
3. Parse the version + variant out of the executable name.
4. Parse a numeric `image_size` out of the image filename (best-effort: first integer in the name).

In [2]:
def parse_executable(name: str):
    """
    Returns (version, variant) given an executable name like 'denoise_1_4'.
    'denoise_1'   -> (1, 0)        base version, no variant
    'denoise_1_4' -> (1, 4)        version 1, variant 4
    Anything that doesn't match returns (None, None).
    """
    m = re.fullmatch(r'denoise_(\d+)(?:_(\d+))?', name)
    if not m:
        return (None, None)
    version = int(m.group(1))
    variant = int(m.group(2)) if m.group(2) is not None else 0
    return (version, variant)


# Hardcoded lookup: noise filename -> (width, height) in pixels.
# Edit this table if you add more test images.  The x-axis of the
# scalability plots uses width * height (total pixels), which is the
# right axis for GPU work since each thread handles one pixel.
IMAGE_RESOLUTIONS = {
    'noise_640.ppm':  (640,  426),
    'noise_1280.ppm': (1280, 853),
    'noise_1920.ppm': (1920, 1280),
    'noise_2k.ppm':   (2560, 1440),
    'noise_5k.ppm':   (5184, 3456),
}


def parse_image_size(name: str):
    """
    Total pixel count for a known noise file.  Returns None for any name
    not in IMAGE_RESOLUTIONS so it's excluded from scalability plots
    rather than silently plotted at a wrong x-value.
    """
    res = IMAGE_RESOLUTIONS.get(name)
    return res[0] * res[1] if res else None


csv_paths = sorted(DATA_DIR.glob('denoise_*.csv'))
if not csv_paths:
    raise FileNotFoundError(f"No denoise_*.csv files found in {DATA_DIR.resolve()}")

frames = []
for p in csv_paths:
    df = pd.read_csv(p)
    df['source_csv'] = p.name
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)

# Filter failed runs
ok = raw[raw['status'] == 'ok'].copy()
n_dropped = len(raw) - len(ok)
if n_dropped:
    print(f"Dropped {n_dropped} failed run(s)")

# Parse executable
parsed = ok['executable'].apply(parse_executable)
ok['version'] = parsed.apply(lambda t: t[0])
ok['variant'] = parsed.apply(lambda t: t[1])

# Parse image size
ok['image_size'] = ok['image'].apply(parse_image_size)

print(f"Loaded {len(csv_paths)} CSV file(s), {len(ok)} successful runs.")
print(f"Executables seen: {sorted(ok['executable'].unique())}")
print(f"Images seen     : {sorted(ok['image'].unique())}")

FileNotFoundError: No denoise_*.csv files found in /home/luca/Computer-Architecture-Project

## 2. Aggregate per (executable, image)

Mean ± std across reps. Median is also kept as a robustness check — if mean and median diverge sharply, that's a hint that one rep was a cold-start outlier.

In [ ]:
agg = (
    ok.groupby(['executable', 'version', 'variant', 'image', 'image_size'],
               dropna=False)
      .agg(mean_ms=('total_ms', 'mean'),
           std_ms=('total_ms', 'std'),
           median_ms=('total_ms', 'median'),
           n_reps=('total_ms', 'size'))
      .reset_index()
)
agg = agg.sort_values(['version', 'variant', 'image_size']).reset_index(drop=True)
agg

## 3. Per-version scalability

One subplot per algorithm version. Each line is a variant within that version (variant 0 = the base, no-tweak run).

The x-axis uses the integer extracted from the image filename. If your filenames don't have a clean integer (e.g. `noise_5k.ppm`), those rows are skipped here — adjust `parse_image_size` above if needed.

In [ ]:
def plot_version_scalability(version: int, ax=None):
    """Plot all variants of a single algorithm version as lines vs image size."""
    sub = agg[(agg['version'] == version) & agg['image_size'].notna()].copy()
    if sub.empty:
        if ax is not None:
            ax.set_title(f'Version {version}: no scalable data')
        return

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4.5))

    # Plot each variant as its own line
    for variant, vsub in sub.groupby('variant'):
        vsub = vsub.sort_values('image_size')
        label = f'denoise_{version}' if variant == 0 else f'denoise_{version}_{variant}'
        ax.errorbar(
            vsub['image_size'], vsub['mean_ms'],
            yerr=vsub['std_ms'].fillna(0),
            marker='o', capsize=3, label=label,
        )

    ax.set_xlabel('image size (total pixels = w × h)')
    ax.set_ylabel('total GPU time (ms)')
    ax.set_title(f'Version {version}: scalability')
    ax.legend(loc='best', fontsize=9)


versions = sorted(v for v in agg['version'].dropna().unique())
for v in versions:
    plot_version_scalability(int(v))
    plt.tight_layout()
    plt.show()

## 4. Cross-version comparison on a single image

Pick the image you want to compare on. The bar chart shows mean total time per executable, ordered by version then variant, with std as error bars. Lower is better.

In [ ]:
# ── Configure this ──────────────────────────────────────────────────────────
COMPARE_IMAGE = 'noise_1280.ppm'   # change to whichever image you want
# ────────────────────────────────────────────────────────────────────────────

def plot_image_comparison(image_name: str, ax=None):
    sub = agg[agg['image'] == image_name].copy()
    if sub.empty:
        available = sorted(agg['image'].unique())
        raise ValueError(
            f"No rows for image '{image_name}'. Available images: {available}"
        )

    sub = sub.sort_values(['version', 'variant'])
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(6, 0.6 * len(sub) + 3), 4.5))

    xpos = np.arange(len(sub))
    ax.bar(
        xpos, sub['mean_ms'],
        yerr=sub['std_ms'].fillna(0),
        capsize=4,
    )
    ax.set_xticks(xpos)
    ax.set_xticklabels(sub['executable'], rotation=30, ha='right')
    ax.set_ylabel('total GPU time (ms)')
    ax.set_title(f'Cross-version comparison on {image_name}')

    # Annotate each bar with its mean value
    for x, v in zip(xpos, sub['mean_ms']):
        ax.text(x, v, f'{v:.2f}', ha='center', va='bottom', fontsize=8)


plot_image_comparison(COMPARE_IMAGE)
plt.tight_layout()
plt.show()

## 5. (Optional) Cross-version comparison on every image

Loop over every image and produce one bar chart per image — useful when you want to scan all comparisons quickly.

In [ ]:
for img in sorted(agg['image'].unique()):
    plot_image_comparison(img)
    plt.tight_layout()
    plt.show()

## 6. (Optional) Speedup table

Treat `denoise_1` (the naive baseline) as version 1.0 and compute speedup factors for every other executable, per image.

In [ ]:
baseline = agg[agg['executable'] == 'denoise_1'][['image', 'mean_ms']].rename(
    columns={'mean_ms': 'baseline_ms'}
)

speedup = agg.merge(baseline, on='image', how='left')
speedup['speedup_vs_denoise_1'] = speedup['baseline_ms'] / speedup['mean_ms']
speedup = speedup[['executable', 'image', 'mean_ms', 'baseline_ms', 'speedup_vs_denoise_1']]
speedup.sort_values(['image', 'speedup_vs_denoise_1'], ascending=[True, False])